In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalJobSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# Configuration is handled inside TotalJobSpendsApp.__init__
# to follow the dependency-injection pattern.

In [ ]:
# =======================================================
# Total Job Spends Client
# =======================================================
class TotalJobSpendsClient:

    TABLE_NAME = "dbspend360_total_job_spends"

    def __init__(
        self,
        audit_table: str,
        cloud_cost_table: str,
        databricks_cost_table: str,
        total_job_spends_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.cloud_cost_table = cloud_cost_table
        self.databricks_cost_table = databricks_cost_table
        self.total_job_spends_table = total_job_spends_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalJobSpendsClient")

    def _log_errors(self, dbu_df, cloud_df):
        dbu_only = (
            dbu_df.alias("d")
            .join(
                cloud_df.alias("a"),
                on=(
                    (F.col("d.cluster_id") == F.col("a.cluster_id")) &
                    (F.col("d.usage_date") == F.col("a.cost_incurred_date"))
                ),
                how="left_anti"
            )
        )

        if not dbu_only.isEmpty():
            dbu_err = (
                dbu_only
                .select(
                    F.lit("DBR_DBU").alias("source_system"),
                    F.lit("NO_MATCH_cloud_COST").alias("error_type"),
                    F.col("d.cluster_id").alias("cluster_id"),
                    "job_id",
                    "run_id",
                    "usage_date",
                    F.col("d.currency").alias("currency"),
                    F.lit("No matching cloud VM cost row for this DBU usage").alias("error_detail"),
                    F.to_json(F.struct("d.*")).alias("raw_record"),
                )
                .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
            )
            _safe_append(dbu_err, self.error_log_table)

        cloud_only = (
            cloud_df.alias("a")
            .join(
                dbu_df.alias("d"),
                on=(
                    (F.col("a.cluster_id") == F.col("d.cluster_id")) &
                    (F.col("a.cost_incurred_date") == F.col("d.usage_date"))
                ),
                how="left_anti"
            )
        )

        if not cloud_only.isEmpty():
            cloud_err = (
                cloud_only
                .select(
                    F.lit("cloud_COST").alias("source_system"),
                    F.lit("NO_MATCH_DBR_DBU").alias("error_type"),
                    F.col("a.cluster_id").alias("cluster_id"),
                    F.lit(None).cast("string").alias("job_id"),
                    F.lit(None).cast("string").alias("run_id"),
                    F.lit(None).cast("date").alias("usage_date"),
                    F.col("a.currency").alias("currency"),
                    F.lit("No matching DBR DBU cost row for this cloud VM cost").alias("error_detail"),
                    F.to_json(F.struct("a.*")).alias("raw_record"),
                )
                .withColumn("created_at", F.lit(datetime.now(timezone.utc)))
            )
            _safe_append(cloud_err, self.error_log_table)

    def build_total_job_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(f"Building dbspend360_total_job_spends for {start_dt} → {end_dt}")

            ensure_cost_columns(self.total_job_spends_table, logger=self.logger)

            cloud_df = (
                spark.table(self.cloud_cost_table)
                    .alias("cc")
                    .filter(
                        (F.col("cost_incurred_date") >= F.lit(start_dt)) &
                        (F.col("cost_incurred_date") <= F.lit(end_dt))
                    )
            )
            dbu_df = (
                spark.table(self.databricks_cost_table)
                    .alias("dbu")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if dbu_df.limit(1).count() == 0:
                self.logger.info("No DBU rows in this date window; nothing to join.")
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window",
                )
                return

            cc_columns = {c.name for c in spark.table(self.cloud_cost_table).schema}
            has_segmented = "compute_cost" in cc_columns
            has_other = "other_cost" in cc_columns

            joined = dbu_df.join(
                cloud_df,
                on=(
                    (dbu_df["cluster_id"] == cloud_df["cluster_id"]) &
                    (dbu_df["usage_date"] == cloud_df["cost_incurred_date"])
                ),
                how="inner"
            )

            joined = joined.withColumn(
                "final_currency",
                F.coalesce(F.col("dbu.currency"), F.col("cc.currency"))
            )

            joined = joined.withColumn(
                "cloud_cost",
                F.col("cc.cloud_cost")
            )

            select_cols = [
                F.col("dbu.cluster_id").alias("cluster_id"),
                "job_id",
                "run_id",
                "usage_date",
                F.col("cloud_cost").alias("cloud_cost"),
                F.col("databricks_cost"),
                F.col("final_currency").alias("currency"),
            ]

            if has_segmented:
                select_cols.extend([
                    F.col("cc.compute_cost").alias("compute_cost"),
                    F.col("cc.storage_cost").alias("storage_cost"),
                    F.col("cc.network_cost").alias("network_cost"),
                ])
            if has_other:
                select_cols.append(F.col("cc.other_cost").alias("other_cost"))

            final_df = joined.select(*select_cols)

            if not has_segmented:
                final_df = (
                    final_df
                    .withColumn("compute_cost", F.lit(None).cast("double"))
                    .withColumn("storage_cost", F.lit(None).cast("double"))
                    .withColumn("network_cost", F.lit(None).cast("double"))
                )
            if not has_other:
                final_df = final_df.withColumn("other_cost", F.lit(None).cast("double"))

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("cloud_cost"), F.lit(0.0))
                    + F.coalesce(F.col("databricks_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"cluster_id": "string", "job_id": "string", "run_id": "string",
                 "usage_date": "date", "cloud_cost": "double", "databricks_cost": "double"},
                self.total_job_spends_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["cloud_cost", "databricks_cost", "total_cost",
                 "compute_cost", "storage_cost", "network_cost", "other_cost"],
                self.total_job_spends_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.total_job_spends_table, self.logger)

            target = DeltaTable.forName(spark, self.total_job_spends_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    "t.cluster_id = s.cluster_id AND t.job_id = s.job_id "
                    "AND t.run_id = s.run_id AND t.usage_date = s.usage_date",
                )
                .whenMatchedUpdate(set={
                    "cloud_cost": "s.cloud_cost",
                    "compute_cost": "s.compute_cost",
                    "storage_cost": "s.storage_cost",
                    "network_cost": "s.network_cost",
                    "other_cost": "s.other_cost",
                    "databricks_cost": "s.databricks_cost",
                    "total_cost": "s.total_cost",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "cluster_id": "s.cluster_id",
                    "job_id": "s.job_id",
                    "run_id": "s.run_id",
                    "usage_date": "s.usage_date",
                    "cloud_cost": "s.cloud_cost",
                    "compute_cost": "s.compute_cost",
                    "storage_cost": "s.storage_cost",
                    "network_cost": "s.network_cost",
                    "other_cost": "s.other_cost",
                    "databricks_cost": "s.databricks_cost",
                    "currency": "s.currency",
                    "total_cost": "s.total_cost",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            get_merge_metrics(self.total_job_spends_table, self.logger)

            validate_post_merge(
                self.total_job_spends_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # Error logging should not impact pipeline success
            try:
                self._log_errors(dbu_df, cloud_df)
            except Exception as e:
                self.logger.exception(
                    f"Error logging failed, but pipeline succeeded: {e}"
                )

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", row_count, "")
            self.logger.info(
                f"Merged {row_count} rows into {self.total_job_spends_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalJobSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalJobSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            cloud_cost_table=build_table_fqn(catalog, schema, "dbspend360_cloud_cost_explorer"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_dbu_cost"),
            total_job_spends_table=build_table_fqn(catalog, schema, "dbspend360_total_job_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_job_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalJobSpendsApp()
app.run()